In [0]:
# ============================================================
# MyCredit-IQ | 02_bronze_macro_features.py
# BNM macroeconomic features (2020–2024)
# Joins to loans_bronze via origination_ym
# ============================================================

import pandas as pd
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()

# ── Monthly date spine (2020-01 to 2024-12) ──
months = pd.date_range("2020-01-01", "2024-12-01", freq="MS")

In [0]:
# ── OPR: actual BNM rate decisions ──
def get_opr(d):
    if d < pd.Timestamp("2020-05-05"):
        return 2.75
    elif d < pd.Timestamp("2020-07-07"):
        return 2.00
    elif d < pd.Timestamp("2020-09-10"):
        return 1.75
    elif d < pd.Timestamp("2022-05-11"):
        return 1.75
    elif d < pd.Timestamp("2022-07-06"):
        return 2.00
    elif d < pd.Timestamp("2022-09-08"):
        return 2.25
    elif d < pd.Timestamp("2023-05-03"):
        return 2.75
    else:
        return 3.00

In [0]:
# ── GDP growth YoY ──
gdp_yoy = {
    2020: -5.6,
    2021: 3.1,
    2022: 8.7,
    2023: 3.6,
    2024: 4.4
}

# ── Household debt-to-GDP ──
hh_debt_gdp = {
    2020: 93.1,
    2021: 89.6,
    2022: 84.5,
    2023: 83.8,
    2024: 84.2
}

# ── CPI YoY inflation ──
cpi_inflation = {
    2020: -1.2,
    2021: 2.5,
    2022: 3.3,
    2023: 2.5,
    2024: 1.8
}

# ── Unemployment rate ──
unemployment = {
    2020: 4.5,
    2021: 4.6,
    2022: 3.7,
    2023: 3.4,
    2024: 3.2
}

In [0]:
# ── Build macro table ──
macro = pd.DataFrame({
    "origination_ym": [d.strftime("%Y-%m") for d in months],
    "origination_year": [d.year for d in months],
    "origination_month": [d.month for d in months],
    "opr_rate": [get_opr(d) for d in months],
    "gdp_growth_yoy": [gdp_yoy[d.year] for d in months],
    "hh_debt_gdp_ratio": [hh_debt_gdp[d.year] for d in months],
    "cpi_inflation_yoy": [cpi_inflation[d.year] for d in months],
    "unemployment_rate": [unemployment[d.year] for d in months],
    "economic_environment": ["recession" if gdp_yoy[d.year] < 0 else "slow_growth" if gdp_yoy[d.year] < 4 else "expansion" for d in months]
})

In [0]:
# ── Validation ──
print("── Macro Features Table ──")
print(macro.to_string(index=False))
print(f"\nRows: {len(macro)} months ({macro['origination_ym'].min()} to {macro['origination_ym'].max()})")
print(f"OPR range: {macro['opr_rate'].min()}% - {macro['opr_rate'].max()}%")
print(f"\nEconomic environment distribution:")
print(macro['economic_environment'].value_counts())

In [0]:
# ── Write to Delta ──
sdf = spark.createDataFrame(macro)

(sdf.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("my_living_index.credit.macro_bronze"))

print(f"\nWritten {len(macro)} rows to my_living_index.credit.macro_bronze")
sdf.printSchema()